# 02 — Model Evaluation, Calibration & Decision Curves

Run after `python scripts/train_pipeline.py`.

This notebook loads the persisted calibrated model and produces the core evaluation figures and tables used in the project report.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data.loader import load_pima_dataset
from src.data.preprocessing import create_clinical_features, train_val_test_split
from src.evaluation.metrics import compute_classification_metrics, bootstrap_auc_ci
from src.evaluation.decision_curve import decision_curve_analysis
from src.utils.config import load_config
from src.utils.reproducibility import set_seed

set_seed(42)
config = load_config()
sns.set_theme(style="whitegrid", context="paper")

In [ ]:
models_dir = Path(config["paths"]["models"])
model = joblib.load(models_dir / "best_model.joblib")
feature_cols = joblib.load(models_dir / "feature_names.joblib")

df = create_clinical_features(load_pima_dataset())
X = df[feature_cols].fillna(df[feature_cols].median())
y = df["Outcome"]
combined = X.copy()
combined["Outcome"] = y
_, _, X_test, _, _, y_test = train_val_test_split(
    combined, target="Outcome", random_state=42
)

proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)
metrics = compute_classification_metrics(y_test.values, pred, proba)
auc, lo, hi = bootstrap_auc_ci(y_test.values, proba)
print(f"ROC-AUC: {auc:.4f} (95% CI {lo:.4f}–{hi:.4f})")
pd.Series(metrics).round(4)

## Clinical reading of metrics

- **Sensitivity** (recall) prioritizes detection of true diabetes cases — important when the cost of a missed diagnosis is high.
- **Specificity** limits false positives that would trigger unnecessary confirmatory testing.
- **PR-AUC** is more informative than ROC-AUC under class imbalance.
- **Brier score** quantifies overall probability quality (lower is better).
- Bootstrap confidence intervals communicate uncertainty due to finite sample size.